In [9]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from rich import print as rprint
##统一使用init_chat_model（）完成模型的初始化
import os
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from typing import Literal

from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from pydantic_core.core_schema import json_schema
from rich import print as rprint

#1 读取，env文件中的配置信息，相关的环境变量以.env文件中的优先
load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#2 初始化大模型
model= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=3,
)
class WeatherInput(BaseModel):
    #如果在函数声明处指定了参数以及参数类型，并且启用了args_schema，那么则以args_schema指定的类型为准
    city: str=Field(
        description="具体的城市名称，例如'北京市'、'重庆市'、''天津市",
        default="北京市",
        min_length=2,
        max_length=10,
    )
    unit:Literal["celsius","fahrenheit","kelvin"]=Field(
        description="描述气温的单位，必须是摄氏度，凯文等单位"
    )
    if_forcast:bool=Field(
        default=False,
        description="是否预测未来1天的天气"
    )
#3 声明1个函数（工具）
@tool(description="获取具体城市的天气",args_schema=WeatherInput)
def get_weather(city:str,unit:str,if_forcast:bool):
    result= f"当天{city}天气晴朗,9-16{unit},温度适宜，适合出门晒太阳"
    if if_forcast:
        result+="但是明天可能有雨，记得带伞"
    return result
#4 将函数绑定在模型上
model_with_tools=model.bind_tools(
    [get_weather]
)
response=convert_to_openai_tool(get_weather)
# rprint(response)
#5 维护1个消息列表
messages=[HumanMessage("今天重庆的天气怎样，明天呢？")]
#6 调用模型 得到1个响应消息 是AIMessage消息对象
response=model_with_tools.invoke(messages)
messages.append(response)
# for msg in messages:
#     msg.pretty_print()
#7 获取响应中的tool_calls字段信息
tool_calls=response.tool_calls
for tool_call in tool_calls:
    if tool_call["name"]=="get_weather":
        #8 因为大模型不能调用工具，所以此时手动调用工具 调用完以后，返回的是ToolMessage消息对象

        tool_message=get_weather.invoke(tool_call)
        messages.append(tool_message)
# for msg in messages:
#     msg.pretty_print()
# 10 最后1次调用大模型 返回的是1个AIMessage
final_response=model.invoke(messages)
# 11 添加到消息列表中
messages.append(final_response)

# 12 遍历消息列表
for msg in messages:
    msg.pretty_print()


================================ Human Message =================================

今天重庆的天气怎样，明天呢？
================================== Ai Message ==================================

我来帮您查询重庆今天和明天的天气情况。
Tool Calls:
  get_weather (call_33dffd398ee74aaf8577bc2d)
 Call ID: call_33dffd398ee74aaf8577bc2d
  Args:
    city: 重庆市
    unit: celsius
    if_forcast: True
================================= Tool Message =================================
Name: get_weather

当天重庆市天气晴朗,9-16celsius,温度适宜，适合出门晒太阳但是明天可能有雨，记得带伞
================================== Ai Message ==================================

今天重庆的天气是**晴朗**的，气温在9-16摄氏度之间，温度适宜，适合出门晒太阳。

明天可能有雨，记得带伞哦。


In [ ]:
for msg in response['messages']:
    if isinstance(msg, AIMessage):
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            print("还在调用工具...")
        else:
            print("完成！最终答案: ", msg.content)